In [ ]:
%pip install transformers datasets huggingface_hub accelerate

In [ ]:
%%python --version

In [ ]:
%pip install torch torchvision transformers datasets huggingface_hub

In [ ]:
import torch

# This tells PyTorch to use the GPU if it's available, otherwise fallback to CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"You are currently using a: {device.upper()}")

In [ ]:
import torchvision.models as models

# 1. Load the ResNet-18 model. "weights='DEFAULT'" loads a pre-trained version.
resnet = models.resnet18(weights='DEFAULT')
print(type(resnet))

# 2. Move the model to our device (GPU)
resnet = resnet.to(device)
resnet.eval()  # Set the model to evaluation mode (important for inference)

print("ResNet successfully loaded and moved to:", device)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

# We use streaming=True to avoid downloading 100+ GB of data!
print("Connecting to ImageNet-1k...")
dataset = load_dataset("imagenet-1k", split="train", streaming=True)
print(type(dataset))

# Look at the very first image in the dataset
iterator = iter(dataset)
first_sample = next(iterator)

print("\nDataset Labels:", first_sample['label'])
first_sample['image']

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests

model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.to(device)

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

text_options = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

inputs = processor(text=text_options, images=image, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}
outputs = clip_model(**inputs)

logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

print("Probability it is a cat:", probs[0][0].item())
print("Probability it is a dog:", probs[0][1].item())
print("Probability it is a car:", probs[0][2].item())

In [ ]:
from torchvision import transforms

# ResNet requires images to be resized and normalized in a very specific way
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Process the image and move it to the GPU
clip_model = clip_model.to(device)
input_tensor = inputs["pixel_values"].to(device)
print(input_tensor.shape)  # Should be [1, 3, 224, 224]

# Run the model
with torch.inference_mode():
    image_features = clip_model.get_image_features(pixel_values=input_tensor)
    output = image_features.pooler_output


print(output.shape) # Should be [1, 1000]

# Get the highest predicted score
predicted_class = output.argmax(dim=1).item()
print(f"ResNet predicts this is class ID: {predicted_class}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful, concise AI assistant."},
    {"role": "user", "content": "What is a Large Language Model? Explain it in one sentence."}
]

text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text_prompt], return_tensors="pt").to(device)

with torch.inference_mode():
    generated_ids = llm_model.generate(**inputs, max_new_tokens=50)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(f"Qwen says: {response}")

In [ ]:
print(device)

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
blip_id = "Salesforce/blip2-opt-2.7b"

processor = Blip2Processor.from_pretrained(blip_id)
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    blip_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

prompt = "Question: What is the main subject of this image and what is happening? Answer:"

inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, torch.float16)

with torch.inference_mode():
    generated_ids = blip_model.generate(**inputs, max_new_tokens=30)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
print(f"BLIP-2 Answer: {generated_text}")